# 🧠 Project 5 — MemoryMesh: Memory-Enabled Conversational Agent

**Core Concept:** Short term + long term memory for personalized AI conversations

### Architecture
User Message
      │
      ▼
Short Term Memory (current session)
      +
Long Term Memory (persistent facts)
      │
      ▼
LLM Response
      │
      ▼
Memory Extraction → Save Important Facts

Install

In [1]:
!pip install -q langchain langchain-groq langchain-core loguru

API Key

In [2]:
import os
os.environ["GROQ_API_KEY"] = "your_actual_groq_key_here"

All Setup In One Block

In [3]:
import os
import json
from datetime import datetime, timezone
from loguru import logger
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import sys

logger.remove()
logger.add(sys.stdout, format="{time:HH:mm:ss} | {level} | {message}", level="DEBUG")

# ── Short Term Memory ────────────────────────────────────────
class ShortTermMemory:
    def __init__(self, max_messages: int = 10):
        self.messages = []
        self.max_messages = max_messages

    def add(self, role: str, content: str):
        self.messages.append({
            "role": role,
            "content": content,
            "timestamp": datetime.now(timezone.utc).isoformat()
        })
        if len(self.messages) > self.max_messages:
            self.messages = self.messages[-self.max_messages:]

    def get_history(self) -> str:
        if not self.messages:
            return "No conversation history yet."
        history = []
        for msg in self.messages:
            role = "User" if msg["role"] == "user" else "Assistant"
            history.append(f"{role}: {msg['content']}")
        return "\n".join(history)

    def clear(self):
        self.messages = []
        logger.info("Short term memory cleared")


# ── Long Term Memory ─────────────────────────────────────────
class LongTermMemory:
    def __init__(self):
        self.facts = {}
        self.interaction_count = 0

    def save_fact(self, key: str, value: str):
        self.facts[key] = {
            "value": value,
            "saved_at": datetime.now(timezone.utc).isoformat()
        }
        logger.info(f"Long term memory saved: {key} = {value}")

    def get_fact(self, key: str) -> str:
        if key in self.facts:
            return self.facts[key]["value"]
        return None

    def get_all_facts(self) -> str:
        if not self.facts:
            return "No long term memories yet."
        facts_list = []
        for key, data in self.facts.items():
            facts_list.append(f"- {key}: {data['value']}")
        return "\n".join(facts_list)

    def increment_interaction(self):
        self.interaction_count += 1

    def get_summary(self) -> dict:
        return {
            "total_facts": len(self.facts),
            "interaction_count": self.interaction_count,
            "facts": self.facts
        }


# ── Memory Extractor ─────────────────────────────────────────
class MemoryExtractor:
    def __init__(self, llm):
        self.llm = llm
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a memory extraction agent.
Extract important personal facts from the conversation that should be remembered long term.

Extract facts like:
- User's name
- User's job or profession
- User's location or city
- User's preferences and interests
- User's goals or projects
- Important dates or events

Respond ONLY in this JSON format with no extra text:
{{"facts": {{"key": "value", "key2": "value2"}}}}

If no important facts to extract, respond with:
{{"facts": {{}}}}"""),
            ("human", "Extract facts from this conversation:\n\n{conversation}")
        ])

    def extract(self, conversation: str) -> dict:
        try:
            chain = self.prompt | self.llm
            response = chain.invoke({"conversation": conversation})
            raw = response.content.strip()

            if "```" in raw:
                import re
                raw = re.sub(r"```(?:json)?", "", raw).strip()

            data = json.loads(raw)
            return data.get("facts", {})
        except Exception as e:
            logger.warning(f"Memory extraction failed: {e}")
            return {}


# ── MemoryMesh Agent ─────────────────────────────────────────
class MemoryMeshAgent:
    def __init__(self):
        self.llm = ChatGroq(
            model="llama-3.3-70b-versatile",
            temperature=0.7,
            api_key=os.environ["GROQ_API_KEY"]
        )
        self.short_term = ShortTermMemory(max_messages=10)
        self.long_term = LongTermMemory()
        self.extractor = MemoryExtractor(self.llm)

        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are MemoryMesh, a personalized AI assistant with memory.
You remember past conversations and personal details about the user.

Long term memories about this user:
{long_term_facts}

Use this information to give personalized responses.
Be warm, helpful, and reference what you know about the user when relevant."""),
            ("human", """Conversation history:
{conversation_history}

Current message: {user_message}

Respond naturally and personally:""")
        ])

        self.chain = self.prompt | self.llm
        logger.info("MemoryMesh agent initialized")

    def chat(self, user_message: str) -> str:
        logger.info(f"User: {user_message[:50]}")

        self.short_term.add("user", user_message)
        self.long_term.increment_interaction()

        long_term_facts = self.long_term.get_all_facts()
        conversation_history = self.short_term.get_history()

        response = self.chain.invoke({
            "long_term_facts": long_term_facts,
            "conversation_history": conversation_history,
            "user_message": user_message
        })

        assistant_message = response.content
        self.short_term.add("assistant", assistant_message)

        extracted_facts = self.extractor.extract(
            f"User: {user_message}\nAssistant: {assistant_message}"
        )

        for key, value in extracted_facts.items():
            self.long_term.save_fact(key, value)

        logger.info(f"Extracted {len(extracted_facts)} facts from conversation")
        return assistant_message

    def get_memory_status(self) -> dict:
        return {
            "short_term_messages": len(self.short_term.messages),
            "long_term_facts": self.long_term.get_summary(),
            "interaction_count": self.long_term.interaction_count
        }

    def display_memories(self):
        print("\n========== MEMORY STATUS ==========")
        print(f"Short Term Messages: {len(self.short_term.messages)}")
        print(f"Interaction Count  : {self.long_term.interaction_count}")
        print(f"\nLong Term Facts:")
        print(self.long_term.get_all_facts())
        print("="*35)

agent = MemoryMeshAgent()
print("MemoryMesh agent ready")

23:39:57 | INFO | MemoryMesh agent initialized
MemoryMesh agent ready


Conversation Test 1: Building Memory

In [4]:
print("========== CONVERSATION: Building Memory ==========\n")

conversations = [
    "Hi! My name is Murali and I'm an AI engineer from Hyderabad.",
    "I'm currently working on a 12-project agentic AI portfolio.",
    "My favorite programming language is Python and I love LangChain.",
    "I'm targeting AI startups for my job search.",
    "What do you remember about me so far?"
]

for message in conversations:
    print(f"User: {message}")
    response = agent.chat(message)
    print(f"Assistant: {response}")
    print("---")

========== CONVERSATION: Building Memory ==========

User: Hi! My name is Murali and I'm an AI engineer from Hyderabad.
23:39:57 | INFO | User: Hi! My name is Murali and I'm an AI engineer from 
23:39:57 | INFO | Long term memory saved: name = Murali
23:39:57 | INFO | Long term memory saved: job = AI engineer
23:39:57 | INFO | Long term memory saved: location = Hyderabad
23:39:57 | INFO | Extracted 3 facts from conversation
Assistant: We've already met, Murali. I remember you're an AI engineer from Hyderabad. What brings you back here today? Is there something specific you'd like to discuss or ask about AI engineering?
---
User: I'm currently working on a 12-project agentic AI portfolio.
23:39:57 | INFO | User: I'm currently working on a 12-project agentic AI p
23:39:58 | INFO | Long term memory saved: name = Murali
23:39:58 | INFO | Long term memory saved: job = AI engineer
23:39:58 | INFO | Long term memory saved: location = Hyderabad
23:39:58 | INFO | Long term memory saved: project

Check Memory State

In [5]:
agent.display_memories()


========== MEMORY STATUS ==========
Short Term Messages: 10
Interaction Count  : 5

Long Term Facts:
- name: Murali
- job: AI engineer
- location: Hyderabad
- project: 12-project agentic AI portfolio
- interests: Python, LangChain
- projects: 12-project agentic AI portfolio
- goals: targeting AI startups for job search
- job or profession: AI engineering
- location or city: Hyderabad
- preferences and interests: AI startups, Python, LangChain
- goals or projects: job search, 12-project agentic AI portfolio
- preferences: Python, LangChain


Conversation Test 2: Using Memory

In [6]:
print("========== CONVERSATION: Using Memory ==========\n")

follow_ups = [
    "Can you suggest some AI startups I should apply to?",
    "What Python libraries should I focus on for my portfolio?",
    "Give me a summary of what you know about me."
]

for message in follow_ups:
    print(f"User: {message}")
    response = agent.chat(message)
    print(f"Assistant: {response}")
    print("---")

========== CONVERSATION: Using Memory ==========

User: Can you suggest some AI startups I should apply to?
23:40:01 | INFO | User: Can you suggest some AI startups I should apply to
23:40:03 | INFO | Long term memory saved: name = Murali
23:40:03 | INFO | Long term memory saved: job or profession = AI engineer
23:40:03 | INFO | Long term memory saved: location = Hyderabad
23:40:03 | INFO | Long term memory saved: expertise = Python and LangChain
23:40:03 | INFO | Long term memory saved: portfolio = 12-project agentic AI portfolio
23:40:03 | INFO | Extracted 5 facts from conversation
Assistant: Murali, I'm glad you're taking the next step in your job search. Considering your expertise in AI engineering, experience with Python and LangChain, and the progress you've made on your 12-project agentic AI portfolio, I think you'd be a great fit for several AI startups.

Given your location in Hyderabad, I'd recommend exploring startups in your local community first. There are some exciting co

New Session Test

In [7]:
print("========== NEW SESSION TEST ==========\n")
print("Simulating a new session — clearing short term memory but keeping long term...\n")

agent.short_term.clear()

print("Short term memory cleared.")
print("Long term memory preserved:")
print(agent.long_term.get_all_facts())

print("\nStarting new conversation...")
message = "Hey, do you remember me?"
print(f"User: {message}")
response = agent.chat(message)
print(f"Assistant: {response}")

========== NEW SESSION TEST ==========

Simulating a new session — clearing short term memory but keeping long term...

23:40:06 | INFO | Short term memory cleared
Short term memory cleared.
Long term memory preserved:
- name: Murali
- job: AI engineer
- location: Hyderabad
- project: 12-project agentic AI portfolio
- interests: Python, LangChain, AI, machine learning
- projects: 12-project agentic AI portfolio
- goals: job search, targeting AI startups
- job or profession: AI engineer
- location or city: Hyderabad
- preferences and interests: AI startups, Python, LangChain
- goals or projects: job search, 12-project agentic AI portfolio
- preferences: Python, LangChain, AI, machine learning
- expertise: Python and LangChain
- portfolio: 12-project agentic AI portfolio
- job_interest: AI engineering
- portfolio_goal: 12-project agentic AI portfolio
- current_project: portfolio development

Starting new conversation...
User: Hey, do you remember me?
23:40:06 | INFO | User: Hey, do you r

Memory Summary

In [8]:
print("========== MEMORYMESH SUMMARY ==========\n")
print("Project      : MemoryMesh — Memory-Enabled Conversational Agent")
print("Author       : K Murali Krishna")
print("Model        : Groq LLaMA-3.3-70b-versatile")
print("\nMemory Architecture:")
print("  Short Term : Last 10 messages in current session")
print("  Long Term  : Persistent facts extracted from conversations")
print("\nKey Capabilities:")
print("  ✓ Short term memory — tracks current conversation")
print("  ✓ Long term memory — persists important user facts")
print("  ✓ Automatic fact extraction after every message")
print("  ✓ Personalized responses using stored memories")
print("  ✓ Session reset with memory preservation")
print("\nProduction Concepts Demonstrated:")
print("  ✓ Two-layer memory architecture")
print("  ✓ Automatic memory extraction using LLM")
print("  ✓ Session management")
print("  ✓ Personalization through memory")

status = agent.get_memory_status()
print(f"\nFinal Memory Status:")
print(f"  Total interactions : {status['interaction_count']}")
print(f"  Long term facts    : {status['long_term_facts']['total_facts']}")
print(f"  Short term messages: {status['short_term_messages']}")

========== MEMORYMESH SUMMARY ==========

Project      : MemoryMesh — Memory-Enabled Conversational Agent
Author       : K Murali Krishna
Model        : Groq LLaMA-3.3-70b-versatile

Memory Architecture:
  Short Term : Last 10 messages in current session
  Long Term  : Persistent facts extracted from conversations

Key Capabilities:
  ✓ Short term memory — tracks current conversation
  ✓ Long term memory — persists important user facts
  ✓ Automatic fact extraction after every message
  ✓ Personalized responses using stored memories
  ✓ Session reset with memory preservation

Production Concepts Demonstrated:
  ✓ Two-layer memory architecture
  ✓ Automatic memory extraction using LLM
  ✓ Session management
  ✓ Personalization through memory

Final Memory Status:
  Total interactions : 9
  Long term facts    : 18
  Short term messages: 2
